# Unidad 4 · Kimball (Notebook 2)

## Creacion de fact_ventas (grano: 1 linea por detalle de factura)

Este notebook crea y carga la tabla de hechos `dw.fact_ventas` usando WideWorldImporters como origen y WideWorldImportersDW2 como destino.

Dimensiones usadas:
- dim_tiempo
- dim_cliente
- dim_producto
- dim_vendedor
- dim_ciudad

### Guia de lectura del notebook
Este notebook esta documentado para que puedas seguir el flujo ETL de izquierda a derecha, como una linea de produccion de datos.

**Orden recomendado de lectura/ejecucion:**
1. Conexion y DDL (preparar entorno y estructura).
2. Extraccion (traer transacciones desde WWI).
3. Resolucion de claves (mapear NK -> SK con reglas SCD).
4. Carga incremental (insertar solo nuevas lineas).
5. Verificaciones (calidad, integridad y reconciliacion).

**Convenciones usadas:**
- `src`: datos de origen transaccional.
- `dim_*`: dimensiones del DW.
- `fact_*`: tabla de hechos final.
- `*_nk`: clave natural del origen.
- `*_sk`: clave sustituta del DW.

## 1) Dependencias

Si hace falta instalar paquetes, descomentar:

```python
# %pip install pandas sqlalchemy pyodbc
```

### Que hace esta celda (Dependencias)
- Importa bibliotecas para: conectarse a SQL Server (`pyodbc`, `sqlalchemy`), manipular datos (`pandas`) y construir cadenas de conexion (`urllib`).
- Si el entorno no tiene paquetes instalados, se puede usar `%pip` desde Jupyter.

In [1]:
import urllib

import pandas as pd
import pyodbc
from sqlalchemy import create_engine, text

print('Librerias importadas correctamente')

Librerias importadas correctamente


## 2) Parametros de conexion

### Que hace esta celda (Conexion)
Paso a paso:
1. Define servidor, BD origen y BD destino.
2. Detecta automaticamente driver ODBC 18 o 17.
3. Construye una funcion reutilizable `construir_engine(...)`.
4. Crea dos engines: origen y destino.
5. Ejecuta `SELECT DB_NAME()` para verificar conectividad real.

**Resultado esperado:** mensajes `Conexion OK origen` y `Conexion OK destino`.

In [2]:
SERVIDOR = r'localhost\SQLEXPRESS02'
BD_ORIGEN = 'WideWorldImporters'
BD_DESTINO = 'WideWorldImportersDW2'
USUARIO = 'ingesta_reader'
CLAVE = '123456789'

drivers = pyodbc.drivers()
if 'ODBC Driver 18 for SQL Server' in drivers:
    DRIVER = 'ODBC Driver 18 for SQL Server'
elif 'ODBC Driver 17 for SQL Server' in drivers:
    DRIVER = 'ODBC Driver 17 for SQL Server'
else:
    raise RuntimeError('No se encontro ODBC Driver 17/18 para SQL Server')

def construir_engine(base_datos: str):
    conn_str = (
        f'DRIVER={{{DRIVER}}};'
        f'SERVER={SERVIDOR};'
        f'DATABASE={base_datos};'
        f'UID={USUARIO};'
        f'PWD={CLAVE};'
        'TrustServerCertificate=yes;'
    )
    params = urllib.parse.quote_plus(conn_str)
    return create_engine(f'mssql+pyodbc:///?odbc_connect={params}')

engine_origen = construir_engine(BD_ORIGEN)
engine_destino = construir_engine(BD_DESTINO)

with engine_origen.connect() as c1:
    v1 = c1.execute(text('SELECT DB_NAME() AS bd')).mappings().first()
with engine_destino.connect() as c2:
    v2 = c2.execute(text('SELECT DB_NAME() AS bd')).mappings().first()

print(f'Conexion OK origen : {v1["bd"]}')
print(f'Conexion OK destino: {v2["bd"]}')

Conexion OK origen : WideWorldImporters
Conexion OK destino: WideWorldImportersDW2


## 3) DDL de la tabla de hechos

Diseno Kimball: fact transaccional con grano 1 linea de factura.

### Que hace esta celda (DDL fact_ventas)
- Crea el esquema `dw` si no existe.
- Crea `dw.fact_ventas` con grano **1 fila = 1 linea de factura**.
- Define medidas (`cantidad`, `monto_neto`, `margen_bruto`, etc.).
- Define constraint unica `(numero_factura, numero_linea)` para evitar duplicados de negocio.
- Crea indices por claves foraneas para mejorar consultas analiticas.

In [3]:
sql_ddl_fact = '''
IF NOT EXISTS (SELECT 1 FROM sys.schemas WHERE name = 'dw')
    EXEC('CREATE SCHEMA dw');

IF OBJECT_ID('dw.fact_ventas', 'U') IS NULL
BEGIN
    CREATE TABLE dw.fact_ventas (
        id_fact_venta_sk      BIGINT IDENTITY(1,1) PRIMARY KEY,
        id_tiempo             INT             NOT NULL,
        id_cliente_sk         INT             NOT NULL,
        id_producto_sk        INT             NOT NULL,
        id_vendedor_sk        INT             NOT NULL,
        id_ciudad_sk          INT             NOT NULL,
        numero_factura        INT             NOT NULL,
        numero_linea          INT             NOT NULL,
        cantidad              DECIMAL(18,4)   NOT NULL,
        precio_unitario       DECIMAL(18,4)   NOT NULL,
        monto_neto            DECIMAL(18,2)   NOT NULL,
        monto_impuesto        DECIMAL(18,2)   NULL,
        costo_estimado        DECIMAL(18,2)   NULL,
        margen_bruto          DECIMAL(18,2)   NULL,
        fecha_carga           DATETIME2       NOT NULL DEFAULT SYSDATETIME(),
        CONSTRAINT uq_fact_ventas_doc_linea UNIQUE (numero_factura, numero_linea)
    );

    CREATE NONCLUSTERED INDEX ix_fact_ventas_tiempo   ON dw.fact_ventas(id_tiempo);
    CREATE NONCLUSTERED INDEX ix_fact_ventas_cliente  ON dw.fact_ventas(id_cliente_sk);
    CREATE NONCLUSTERED INDEX ix_fact_ventas_producto ON dw.fact_ventas(id_producto_sk);
    CREATE NONCLUSTERED INDEX ix_fact_ventas_vendedor ON dw.fact_ventas(id_vendedor_sk);
    CREATE NONCLUSTERED INDEX ix_fact_ventas_ciudad   ON dw.fact_ventas(id_ciudad_sk);
END;
'''

with engine_destino.begin() as conn:
    conn.execute(text(sql_ddl_fact))

print('Tabla dw.fact_ventas verificada/creada correctamente')

Tabla dw.fact_ventas verificada/creada correctamente


## 4) Extraccion de lineas de factura desde WWI

### Que hace esta celda (Extraccion + preparacion)
Lectura por bloques:
1. SQL fuente: une `InvoiceLines`, `Invoices` y `Customers`.
2. Carga en `df_fact_src` con `pd.read_sql`.
3. Valida que no venga vacio (si viene vacio, corta con error).
4. Convierte `fecha_factura` a datetime.
5. Deriva `id_tiempo` en formato `YYYYMMDD` para enlazar con `dim_tiempo`.
6. Ordena por factura + linea origen para tener numeracion deterministica.
7. Genera `numero_linea` por factura (`groupby + cumcount + 1`).
8. Calcula `costo_estimado = monto_neto - margen_bruto`.

In [4]:
sql_fact_src = '''
SELECT
    i.InvoiceID                           AS numero_factura,
    il.InvoiceLineID                      AS id_linea_origen,
    CAST(i.InvoiceDate AS date)           AS fecha_factura,
    i.CustomerID                          AS id_cliente_nk,
    il.StockItemID                        AS id_producto_nk,
    i.SalespersonPersonID                 AS id_vendedor_nk,
    c.DeliveryCityID                      AS id_ciudad_nk,
    CAST(il.Quantity AS decimal(18,4))    AS cantidad,
    CAST(il.UnitPrice AS decimal(18,4))   AS precio_unitario,
    CAST(il.ExtendedPrice AS decimal(18,2)) AS monto_neto,
    CAST(il.TaxAmount AS decimal(18,2))   AS monto_impuesto,
    CAST(il.LineProfit AS decimal(18,2))  AS margen_bruto
FROM Sales.InvoiceLines il
JOIN Sales.Invoices i
    ON il.InvoiceID = i.InvoiceID
JOIN Sales.Customers c
    ON i.CustomerID = c.CustomerID;
'''

with engine_origen.connect() as conn:
    df_fact_src = pd.read_sql(sql_fact_src, conn)

if df_fact_src.empty:
    raise RuntimeError('La extraccion de Sales.InvoiceLines devolvio 0 filas')

df_fact_src['fecha_factura'] = pd.to_datetime(df_fact_src['fecha_factura'])
df_fact_src['id_tiempo'] = df_fact_src['fecha_factura'].dt.strftime('%Y%m%d').astype(int)
df_fact_src = df_fact_src.sort_values(['numero_factura', 'id_linea_origen']).copy()
df_fact_src['numero_linea'] = df_fact_src.groupby('numero_factura').cumcount() + 1
df_fact_src['costo_estimado'] = df_fact_src['monto_neto'] - df_fact_src['margen_bruto'].fillna(0)

print(f'Filas extraidas de origen: {len(df_fact_src)}')
df_fact_src.head()

Filas extraidas de origen: 228265


,numero_factura,id_linea_origen,fecha_factura,id_cliente_nk,id_producto_nk,id_vendedor_nk,id_ciudad_nk,cantidad,precio_unitario,monto_neto,monto_impuesto,margen_bruto,id_tiempo,numero_linea,costo_estimado
0,1,1,2013-01-01,832,67,2,13242,10.0,230.0,2645.00,345.00,850.0,20130101,1,1795.00
1,2,2,2013-01-01,803,50,8,33799,9.0,13.0,134.55,17.55,76.5,20130101,1,58.05
2,2,3,2013-01-01,803,10,8,33799,9.0,32.0,331.20,43.20,180.0,20130101,2,151.20
3,3,4,2013-01-01,105,114,7,34208,3.0,30.0,103.50,13.50,24.0,20130101,1,79.50
4,4,5,2013-01-01,57,206,16,18514,96.0,2.7,298.08,38.88,96.0,20130101,1,202.08


In [6]:
df_fact_src.head(20)

,numero_factura,id_linea_origen,fecha_factura,id_cliente_nk,id_producto_nk,id_vendedor_nk,id_ciudad_nk,cantidad,precio_unitario,monto_neto,monto_impuesto,margen_bruto,id_tiempo,numero_linea,costo_estimado
0,1,1,2013-01-01,832,67,2,13242,10.0,230.0,2645.00,345.00,850.0,20130101,1,1795.00
1,2,2,2013-01-01,803,50,8,33799,9.0,13.0,134.55,17.55,76.5,20130101,1,58.05
2,2,3,2013-01-01,803,10,8,33799,9.0,32.0,331.20,43.20,180.0,20130101,2,151.20
3,3,4,2013-01-01,105,114,7,34208,3.0,30.0,103.50,13.50,24.0,20130101,1,79.50
4,4,5,2013-01-01,57,206,16,18514,96.0,2.7,298.08,38.88,96.0,20130101,1,202.08
5,4,6,2013-01-01,57,130,16,18514,5.0,32.0,184.00,24.00,120.0,20130101,2,64.00
6,4,7,2013-01-01,57,50,16,18514,2.0,13.0,29.90,3.90,17.0,20130101,3,12.90
7,5,8,2013-01-01,905,128,3,3425,4.0,32.0,147.20,19.20,96.0,20130101,1,51.20
8,5,9,2013-01-01,905,121,3,3425,3.0,32.0,110.40,14.40,70.5,20130101,2,39.90
9,5,10,2013-01-01,905,155,3,3425,20.0,24.0,552.00,72.00,220.0,20130101,3,332.00


## 5) Resolucion de surrogate keys de dimensiones

Para dimensiones SCD2 (cliente y producto) se resuelve por rango de vigencia usando fecha_factura.

### Que hace esta celda (Resolucion de SK)
Bloque clave de Kimball:
1. Carga cada dimension desde DW.
2. Convierte fechas de vigencia para comparaciones correctas.
3. Hace `merge` con `dim_tiempo` por `id_tiempo`.
4. Hace `merge` con `dim_cliente` por NK y filtra por vigencia (`fecha_factura` dentro del rango SCD2).
5. Hace `merge` con `dim_producto` por NK y tambien filtra por vigencia SCD2.
6. Hace `merge` con `dim_vendedor` y `dim_ciudad` (SCD1, sin rango).
7. Selecciona columnas finales de la fact.
8. Convierte SK a numerico, separa rechazos (SK nula) y registros validos.
9. Convierte SK validas a entero para carga en SQL.

In [7]:
# 1) Cargar dimensiones desde el DW para resolver claves sustitutas (SK).
with engine_destino.connect() as conn:
    # Dimension tiempo (no tiene SCD, se relaciona por id_tiempo directo).
    dim_tiempo = pd.read_sql('SELECT id_tiempo FROM dw.dim_tiempo', conn)

    # Dimensiones SCD2 con bandera actual para fallback cuando no hay match temporal.
    dim_cliente = pd.read_sql('''
        SELECT id_cliente_sk, id_cliente_nk, fecha_inicio_vigencia, fecha_fin_vigencia, es_actual
        FROM dw.dim_cliente
    ''', conn)

    dim_producto = pd.read_sql('''
        SELECT id_producto_sk, id_producto_nk, fecha_inicio_vigencia, fecha_fin_vigencia, es_actual
        FROM dw.dim_producto
    ''', conn)

    # Dimensiones SCD1 (mapeo directo NK -> SK vigente).
    dim_vendedor = pd.read_sql('SELECT id_vendedor_sk, id_vendedor_nk FROM dw.dim_vendedor', conn)
    dim_ciudad = pd.read_sql('SELECT id_ciudad_sk, id_ciudad_nk FROM dw.dim_ciudad', conn)

# 2) Normalizar columnas de vigencia a datetime para comparaciones temporales seguras.
dim_cliente['fecha_inicio_vigencia'] = pd.to_datetime(dim_cliente['fecha_inicio_vigencia'])
dim_cliente['fecha_fin_vigencia'] = pd.to_datetime(dim_cliente['fecha_fin_vigencia'])
dim_producto['fecha_inicio_vigencia'] = pd.to_datetime(dim_producto['fecha_inicio_vigencia'])
dim_producto['fecha_fin_vigencia'] = pd.to_datetime(dim_producto['fecha_fin_vigencia'])

# 3) Definir identificador tecnico por fila para resolver SCD2 sin perder grano.
df_fact_base = df_fact_src.copy()
df_fact_base['_row_id'] = range(len(df_fact_base))

# 4) Resolver cliente SCD2 con fallback a es_actual=1 si no hay match por fecha.
map_cli = df_fact_base[['_row_id', 'id_cliente_nk', 'fecha_factura']].merge(
    dim_cliente,
    on='id_cliente_nk',
    how='left'
)

mask_cli_vigente = (
    (map_cli['fecha_factura'] >= map_cli['fecha_inicio_vigencia'])
    & (map_cli['fecha_factura'] <= map_cli['fecha_fin_vigencia'])
)

cli_vigente = (
    map_cli[mask_cli_vigente]
    .sort_values(['_row_id', 'id_cliente_sk'])
    .drop_duplicates('_row_id')[['_row_id', 'id_cliente_sk']]
)

cli_actual = (
    map_cli[map_cli['es_actual'] == 1]
    .sort_values(['_row_id', 'id_cliente_sk'])
    .drop_duplicates('_row_id')[['_row_id', 'id_cliente_sk']]
)

cli_resuelta = cli_vigente.set_index('_row_id')['id_cliente_sk'].combine_first(
    cli_actual.set_index('_row_id')['id_cliente_sk']
).rename('id_cliente_sk').reset_index()

# 5) Resolver producto SCD2 con el mismo patron.
map_prd = df_fact_base[['_row_id', 'id_producto_nk', 'fecha_factura']].merge(
    dim_producto,
    on='id_producto_nk',
    how='left'
)

mask_prd_vigente = (
    (map_prd['fecha_factura'] >= map_prd['fecha_inicio_vigencia'])
    & (map_prd['fecha_factura'] <= map_prd['fecha_fin_vigencia'])
)

prd_vigente = (
    map_prd[mask_prd_vigente]
    .sort_values(['_row_id', 'id_producto_sk'])
    .drop_duplicates('_row_id')[['_row_id', 'id_producto_sk']]
)

prd_actual = (
    map_prd[map_prd['es_actual'] == 1]
    .sort_values(['_row_id', 'id_producto_sk'])
    .drop_duplicates('_row_id')[['_row_id', 'id_producto_sk']]
)

prd_resuelta = prd_vigente.set_index('_row_id')['id_producto_sk'].combine_first(
    prd_actual.set_index('_row_id')['id_producto_sk']
).rename('id_producto_sk').reset_index()

# 6) Resolver id_tiempo y dimensiones SCD1 por NK.
df_fact = df_fact_base.merge(dim_tiempo, on='id_tiempo', how='left')
df_fact = df_fact.merge(cli_resuelta, on='_row_id', how='left')
df_fact = df_fact.merge(prd_resuelta, on='_row_id', how='left')
df_fact = df_fact.merge(dim_vendedor, on='id_vendedor_nk', how='left')
df_fact = df_fact.merge(dim_ciudad, on='id_ciudad_nk', how='left')

# 7) Seleccionar unicamente columnas objetivo de la tabla de hechos.
columnas_fact = [
    'id_tiempo', 'id_cliente_sk', 'id_producto_sk', 'id_vendedor_sk', 'id_ciudad_sk',
    'numero_factura', 'numero_linea', 'cantidad', 'precio_unitario',
    'monto_neto', 'monto_impuesto', 'costo_estimado', 'margen_bruto',
]

df_fact = df_fact[columnas_fact].copy()

# 8) Forzar tipo numerico en SK para detectar errores de mapeo (valores no convertibles -> NaN).
for c in ['id_tiempo', 'id_cliente_sk', 'id_producto_sk', 'id_vendedor_sk', 'id_ciudad_sk']:
    df_fact[c] = pd.to_numeric(df_fact[c], errors='coerce')

# 9) Separar registros rechazados por SK faltante vs registros listos para carga.
rechazos_sk = df_fact[
    df_fact[['id_tiempo', 'id_cliente_sk', 'id_producto_sk', 'id_vendedor_sk', 'id_ciudad_sk']]
    .isna()
    .any(axis=1)
].copy()

df_fact_ok = df_fact.dropna(
    subset=['id_tiempo', 'id_cliente_sk', 'id_producto_sk', 'id_vendedor_sk', 'id_ciudad_sk']
).copy()

# 10) Convertir SK validas a int para que coincidan con el tipo de SQL Server.
df_fact_ok[['id_tiempo', 'id_cliente_sk', 'id_producto_sk', 'id_vendedor_sk', 'id_ciudad_sk']] = df_fact_ok[[
    'id_tiempo', 'id_cliente_sk', 'id_producto_sk', 'id_vendedor_sk', 'id_ciudad_sk',
]].astype(int)

print(f'Filas candidatas con SK resuelta : {len(df_fact_ok)}')
print(f'Filas rechazadas por SK faltante: {len(rechazos_sk)}')

Filas candidatas con SK resuelta : 228265
Filas rechazadas por SK faltante: 0


In [8]:
df_fact_ok

,id_tiempo,id_cliente_sk,id_producto_sk,id_vendedor_sk,id_ciudad_sk,numero_factura,numero_linea,cantidad,precio_unitario,monto_neto,monto_impuesto,costo_estimado,margen_bruto
0,20130101,226,67,1,13169,1,1,10.0,230.0,2645.00,345.00,1795.00,850.0
1,20130101,581,50,5,33588,2,1,9.0,13.0,134.55,17.55,58.05,76.5
2,20130101,581,10,5,33588,2,2,9.0,32.0,331.20,43.20,151.20,180.0
3,20130101,592,114,4,33996,3,1,3.0,30.0,103.50,13.50,79.50,24.0
4,20130101,325,206,9,18417,4,1,96.0,2.7,298.08,38.88,202.08,96.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
228260,20160531,473,59,10,26418,70510,1,6.0,25.0,172.50,22.50,97.50,75.0
228261,20160531,473,2,10,26418,70510,2,1.0,25.0,28.75,3.75,13.25,15.5
228262,20160531,473,102,10,26418,70510,3,9.0,35.0,362.25,47.25,200.25,162.0
228263,20160531,473,170,10,26418,70510,4,40.0,102.0,4692.00,612.00,2812.00,1880.0


## 6) Carga incremental de fact_ventas

### Que hace esta celda (Carga incremental)
- Lee llaves de negocio ya cargadas en `fact_ventas` (`numero_factura`, `numero_linea`).
- Si la fact esta vacia: inserta todo `df_fact_ok`.
- Si no esta vacia: arma un indice en memoria y filtra solo filas nuevas.
- Inserta solo nuevos registros para evitar reprocesar historico.

**Idea ETL:** idempotencia basica en recargas.

In [11]:
with engine_destino.connect() as conn:
    fact_existente = pd.read_sql('SELECT numero_factura, numero_linea FROM dw.fact_ventas', conn)

if fact_existente.empty:
    df_nuevas = df_fact_ok.copy()
else:
    idx_existente = set(zip(fact_existente['numero_factura'].tolist(), fact_existente['numero_linea'].tolist()))
    idx_actual = list(zip(df_fact_ok['numero_factura'].tolist(), df_fact_ok['numero_linea'].tolist()))
    mascara_nuevas = [k not in idx_existente for k in idx_actual]
    df_nuevas = df_fact_ok[mascara_nuevas].copy()

if not df_nuevas.empty:
    df_nuevas.to_sql('fact_ventas', engine_destino, schema='dw', if_exists='append', index=False)

print(f'Filas nuevas insertadas en fact_ventas: {len(df_nuevas)}')

Filas nuevas insertadas en fact_ventas: 228265


## 7) Verificaciones de consistencia y calidad de datos

### Que hace esta seccion (Calidad y consistencia)
Estas celdas verifican:
1. Volumen total origen vs destino.
2. Integridad referencial (FK huerfanas).
3. Duplicados por llave de negocio.
4. Reglas simples de negocio (cantidades y montos no negativos).
5. Reconciliacion agregada de montos por mes.

### Que hace esta celda (Control de volumen)
Esta verificacion compara el volumen total de filas entre origen y destino:

1. Cuenta filas en `Sales.InvoiceLines` (origen transaccional).
2. Cuenta filas en `dw.fact_ventas` (destino dimensional).
3. Muestra ambos conteos para detectar desbalances grandes.
4. Informa cuantas filas fueron rechazadas en la corrida por SK faltante.

Interpretacion didactica:
- Diferencia esperable en cargas incrementales parciales.
- Diferencia no esperable cuando la corrida pretendia una carga completa.

In [13]:
sql_volumen = '''
SELECT 'fact_ventas' AS tabla, COUNT(*) AS filas FROM dw.fact_ventas
UNION ALL
SELECT 'origen_invoice_lines' AS tabla, COUNT(*) AS filas FROM Sales.InvoiceLines;
'''

with engine_destino.connect() as conn_dw, engine_origen.connect() as conn_src:
    filas_fact = pd.read_sql('SELECT COUNT(*) AS filas FROM dw.fact_ventas', conn_dw)
    filas_src = pd.read_sql('SELECT COUNT(*) AS filas FROM Sales.InvoiceLines', conn_src)

print('Volumen destino vs origen')
display(pd.DataFrame([
    {'tabla': 'fact_ventas', 'filas': int(filas_fact['filas'].iloc[0])},
    {'tabla': 'sales_invoice_lines', 'filas': int(filas_src['filas'].iloc[0])}
]))

print(f'Rechazos por SK faltante en corrida actual: {len(rechazos_sk)}')

Volumen destino vs origen


,tabla,filas
0,fact_ventas,228265
1,sales_invoice_lines,228265


Rechazos por SK faltante en corrida actual: 0


### Que hace esta celda (Control de FK huerfanas)
Este control valida integridad referencial en la fact:

1. Recorre cada FK de `fact_ventas`.
2. Busca si existe en su dimension correspondiente.
3. Suma cuantas quedaron sin match (huerfanas).

Resultado esperado:
- Todos los contadores en 0.
- Si alguno es mayor a 0, hay error de mapeo o de carga de dimensiones.

In [14]:
sql_fk_huerfanas = '''
SELECT
    SUM(CASE WHEN t.id_tiempo IS NULL THEN 1 ELSE 0 END) AS fk_tiempo_huerfana,
    SUM(CASE WHEN c.id_cliente_sk IS NULL THEN 1 ELSE 0 END) AS fk_cliente_huerfana,
    SUM(CASE WHEN p.id_producto_sk IS NULL THEN 1 ELSE 0 END) AS fk_producto_huerfana,
    SUM(CASE WHEN v.id_vendedor_sk IS NULL THEN 1 ELSE 0 END) AS fk_vendedor_huerfana,
    SUM(CASE WHEN ci.id_ciudad_sk IS NULL THEN 1 ELSE 0 END) AS fk_ciudad_huerfana
FROM dw.fact_ventas f
LEFT JOIN dw.dim_tiempo t   ON f.id_tiempo = t.id_tiempo
LEFT JOIN dw.dim_cliente c  ON f.id_cliente_sk = c.id_cliente_sk
LEFT JOIN dw.dim_producto p ON f.id_producto_sk = p.id_producto_sk
LEFT JOIN dw.dim_vendedor v ON f.id_vendedor_sk = v.id_vendedor_sk
LEFT JOIN dw.dim_ciudad ci  ON f.id_ciudad_sk = ci.id_ciudad_sk;
'''

with engine_destino.connect() as conn:
    df_fk_huerfanas = pd.read_sql(sql_fk_huerfanas, conn)

df_fk_huerfanas

,fk_tiempo_huerfana,fk_cliente_huerfana,fk_producto_huerfana,fk_vendedor_huerfana,fk_ciudad_huerfana
0,0,0,0,0,0


### Que hace esta celda (Control de duplicados)
Esta celda verifica duplicados de negocio por combinacion:
- `numero_factura`
- `numero_linea`

Si aparece alguna fila en la salida, la fact tiene duplicados y hay que revisar la logica incremental.

In [15]:
sql_duplicados = '''
SELECT
    numero_factura,
    numero_linea,
    COUNT(*) AS cantidad
FROM dw.fact_ventas
GROUP BY numero_factura, numero_linea
HAVING COUNT(*) > 1
ORDER BY cantidad DESC;
'''

with engine_destino.connect() as conn:
    df_duplicados = pd.read_sql(sql_duplicados, conn)

print(f'Combinaciones duplicadas factura+linea: {len(df_duplicados)}')
df_duplicados.head()

Combinaciones duplicadas factura+linea: 0


,numero_factura,numero_linea,cantidad


### Que hace esta celda (Reglas de negocio minimas)
Controla reglas funcionales basicas:

1. `cantidad` debe ser mayor a 0.
2. `precio_unitario` no debe ser negativo.
3. `monto_neto` no debe ser negativo.

Si algun contador es mayor a 0, hay datos anomalos que requieren analisis.

In [16]:
sql_reglas = '''
SELECT
    SUM(CASE WHEN cantidad <= 0 THEN 1 ELSE 0 END) AS cantidad_no_positiva,
    SUM(CASE WHEN precio_unitario < 0 THEN 1 ELSE 0 END) AS precio_unitario_negativo,
    SUM(CASE WHEN monto_neto < 0 THEN 1 ELSE 0 END) AS monto_neto_negativo
FROM dw.fact_ventas;
'''

with engine_destino.connect() as conn:
    df_reglas = pd.read_sql(sql_reglas, conn)

df_reglas

,cantidad_no_positiva,precio_unitario_negativo,monto_neto_negativo
0,0,0,0


## 8) Reconciliacion de montos origen vs destino (mensual)

Compara montos netos por anio/mes para detectar desbalances en la carga.

### Que hace esta celda (Reconciliacion mensual)
- Agrega `monto_neto` en origen por anio/mes.
- Agrega `monto_neto` en DW por anio/mes usando `dim_tiempo`.
- Une ambos resultados y calcula `diferencia`.
- Si la diferencia es cercana a 0, la carga mensual esta reconciliada.

In [17]:
sql_src_mensual = '''
SELECT
    YEAR(i.InvoiceDate) AS anio,
    MONTH(i.InvoiceDate) AS mes,
    SUM(CAST(il.ExtendedPrice AS decimal(18,2))) AS monto_neto_src
FROM Sales.InvoiceLines il
JOIN Sales.Invoices i
    ON il.InvoiceID = i.InvoiceID
GROUP BY YEAR(i.InvoiceDate), MONTH(i.InvoiceDate);
'''

sql_dw_mensual = '''
SELECT
    t.anio_numero AS anio,
    t.mes_numero AS mes,
    SUM(f.monto_neto) AS monto_neto_dw
FROM dw.fact_ventas f
JOIN dw.dim_tiempo t
    ON f.id_tiempo = t.id_tiempo
GROUP BY t.anio_numero, t.mes_numero;
'''

with engine_origen.connect() as conn_src:
    df_src_mensual = pd.read_sql(sql_src_mensual, conn_src)
with engine_destino.connect() as conn_dw:
    df_dw_mensual = pd.read_sql(sql_dw_mensual, conn_dw)

df_recon = df_src_mensual.merge(df_dw_mensual, on=['anio', 'mes'], how='outer').fillna(0)
df_recon['diferencia'] = (df_recon['monto_neto_dw'] - df_recon['monto_neto_src']).round(2)
df_recon = df_recon.sort_values(['anio', 'mes']).reset_index(drop=True)

print('Reconciliacion mensual origen vs destino')
df_recon

Reconciliacion mensual origen vs destino


,anio,mes,monto_neto_src,monto_neto_dw,diferencia
0,2013,1,4335972.97,4335972.97,0.0
1,2013,2,3193304.60,3193304.60,0.0
2,2013,3,4451081.62,4451081.62,0.0
3,2013,4,4668548.40,4668548.40,0.0
4,2013,5,5080661.05,5080661.05,0.0
5,2013,6,4679392.14,4679392.14,0.0
6,2013,7,5039033.18,5039033.18,0.0
7,2013,8,4020390.18,4020390.18,0.0
8,2013,9,4345897.56,4345897.56,0.0
9,2013,10,4315500.25,4315500.25,0.0


## 9) Muestra final de fact_ventas

### Que hace esta celda (Muestra final)
Consulta una muestra reciente de `dw.fact_ventas` para revisar visualmente:
- llaves sustitutas,
- llaves de negocio,
- medidas principales.

Es una validacion rapida de forma y contenido.

In [18]:
sql_muestra_fact = '''
SELECT TOP 30
    id_fact_venta_sk,
    id_tiempo,
    id_cliente_sk,
    id_producto_sk,
    id_vendedor_sk,
    id_ciudad_sk,
    numero_factura,
    numero_linea,
    cantidad,
    precio_unitario,
    monto_neto,
    margen_bruto
FROM dw.fact_ventas
ORDER BY id_fact_venta_sk DESC;
'''

with engine_destino.connect() as conn:
    df_muestra_fact = pd.read_sql(sql_muestra_fact, conn)

df_muestra_fact

,id_fact_venta_sk,id_tiempo,id_cliente_sk,id_producto_sk,id_vendedor_sk,id_ciudad_sk,numero_factura,numero_linea,cantidad,precio_unitario,monto_neto,margen_bruto
0,228265,20160531,473,130,10,26418,70510,5,8.0,32.0,294.40,192.0
1,228264,20160531,473,170,10,26418,70510,4,40.0,102.0,4692.00,1880.0
2,228263,20160531,473,102,10,26418,70510,3,9.0,35.0,362.25,162.0
3,228262,20160531,473,2,10,26418,70510,2,1.0,25.0,28.75,15.5
4,228261,20160531,473,59,10,26418,70510,1,6.0,25.0,172.50,75.0
5,228260,20160531,15,86,1,926,70509,5,24.0,18.0,496.80,240.0
6,228259,20160531,15,19,1,926,70509,4,3.0,13.0,44.85,25.5
7,228258,20160531,15,215,1,926,70509,3,1.0,1899.0,2183.85,759.0
8,228257,20160531,15,20,1,926,70509,2,5.0,13.0,74.75,42.5
9,228256,20160531,15,25,1,926,70509,1,7.0,13.0,104.65,59.5


## 10) Checklist de cierre de la practica
Usa este checklist al terminar la ejecucion:

- [ ] `df_fact_ok` tiene filas y `rechazos_sk` es razonable.
- [ ] La carga incremental inserto nuevas filas cuando correspondia.
- [ ] No hay FK huerfanas.
- [ ] No hay duplicados por `numero_factura` + `numero_linea`.
- [ ] Las reglas minimas de negocio no reportan anomalias.
- [ ] La reconciliacion mensual tiene diferencias explicables o cercanas a 0.

Si algun punto falla, revisar primero:
1. Vigencias SCD2 en dimensiones.
2. Mapeo NK -> SK.
3. Filtros de carga incremental.

## 11) Consultas de negocio que guian el modelo

Estas consultas responden las preguntas funcionales planteadas al inicio del diseno:

1. Cuanto vendemos por mes, categoria y cliente?
2. Que clientes aportan mayor margen?
3. Como evolucionan las ventas por ciudad y estado?
4. Que vendedor mantiene mejor ticket promedio?
5. Que productos pierden margen por descuentos frecuentes?

In [19]:
sql_q1_ventas_mes_categoria_cliente = '''
SELECT
    t.anio_numero,
    t.mes_numero,
    t.mes_nombre,
    c.categoria_cliente,
    c.nombre_cliente,
    SUM(f.monto_neto) AS ventas_netas,
    SUM(f.cantidad) AS unidades_vendidas,
    SUM(f.margen_bruto) AS margen_bruto
FROM dw.fact_ventas f
JOIN dw.dim_tiempo t
    ON f.id_tiempo = t.id_tiempo
JOIN dw.dim_cliente c
    ON f.id_cliente_sk = c.id_cliente_sk
--where t.anio_numero >= 2020 
--and t.anio_numero <= 2025
GROUP BY
    t.anio_numero,
    t.mes_numero,
    t.mes_nombre,
    c.categoria_cliente,
    c.nombre_cliente
ORDER BY
    t.anio_numero,
    t.mes_numero,
    c.categoria_cliente,
    ventas_netas DESC;
'''

sql_q2_clientes_mayor_margen = '''
SELECT
    c.id_cliente_nk,
    c.nombre_cliente,
    SUM(f.margen_bruto) AS margen_bruto_total,
    SUM(f.monto_neto) AS ventas_netas_totales,
    CASE
        WHEN SUM(f.monto_neto) = 0 THEN NULL
        ELSE (SUM(f.margen_bruto) * 1.0 / SUM(f.monto_neto))
    END AS margen_pct
FROM dw.fact_ventas f
JOIN dw.dim_cliente c
    ON f.id_cliente_sk = c.id_cliente_sk
GROUP BY
    c.id_cliente_nk,
    c.nombre_cliente
ORDER BY
    margen_bruto_total DESC;
'''

sql_q3_evolucion_ventas_ciudad_estado = '''
SELECT
    t.anio_numero,
    t.mes_numero,
    ci.estado_provincia,
    ci.ciudad,
    SUM(f.monto_neto) AS ventas_netas,
    SUM(f.margen_bruto) AS margen_bruto,
    SUM(f.cantidad) AS unidades_vendidas
FROM dw.fact_ventas f
JOIN dw.dim_tiempo t
    ON f.id_tiempo = t.id_tiempo
JOIN dw.dim_ciudad ci
    ON f.id_ciudad_sk = ci.id_ciudad_sk
GROUP BY
    t.anio_numero,
    t.mes_numero,
    ci.estado_provincia,
    ci.ciudad
ORDER BY
    t.anio_numero,
    t.mes_numero,
    ci.estado_provincia,
    ventas_netas DESC;
'''

sql_q4_vendedor_mejor_ticket_promedio = '''
WITH ticket_por_factura AS (
    SELECT
        f.id_vendedor_sk,
        f.numero_factura,
        SUM(f.monto_neto) AS ticket_neto
    FROM dw.fact_ventas f
    GROUP BY
        f.id_vendedor_sk,
        f.numero_factura
)
SELECT
    v.id_vendedor_nk,
    v.nombre_completo,
    AVG(tpf.ticket_neto) AS ticket_promedio,
    COUNT(*) AS cantidad_facturas,
    SUM(tpf.ticket_neto) AS ventas_netas_totales
FROM ticket_por_factura tpf
JOIN dw.dim_vendedor v
    ON tpf.id_vendedor_sk = v.id_vendedor_sk
GROUP BY
    v.id_vendedor_nk,
    v.nombre_completo
ORDER BY
    ticket_promedio DESC;
'''

sql_q5_productos_descuento_y_perdida_margen = '''
SELECT
    p.id_producto_nk,
    p.nombre_producto,
    p.marca,
    COUNT(*) AS lineas_totales,
    SUM(CASE WHEN f.precio_unitario < p.precio_recomendado THEN 1 ELSE 0 END) AS lineas_con_descuento,
    CAST(SUM(CASE WHEN f.precio_unitario < p.precio_recomendado THEN 1 ELSE 0 END) * 100.0 / COUNT(*) AS decimal(10,2)) AS pct_lineas_con_descuento,
    SUM(CASE WHEN f.precio_unitario < p.precio_recomendado THEN (p.precio_recomendado - f.precio_unitario) * f.cantidad ELSE 0 END) AS perdida_ingreso_estimada,
    SUM(f.margen_bruto) AS margen_bruto_total
FROM dw.fact_ventas f
JOIN dw.dim_producto p
    ON f.id_producto_sk = p.id_producto_sk
GROUP BY
    p.id_producto_nk,
    p.nombre_producto,
    p.marca
HAVING
    SUM(CASE WHEN f.precio_unitario < p.precio_recomendado THEN 1 ELSE 0 END) > 0
ORDER BY
    pct_lineas_con_descuento DESC,
    perdida_ingreso_estimada DESC;
'''

with engine_destino.connect() as conn:
    df_q1 = pd.read_sql(sql_q1_ventas_mes_categoria_cliente, conn)
    df_q2 = pd.read_sql(sql_q2_clientes_mayor_margen, conn)
    df_q3 = pd.read_sql(sql_q3_evolucion_ventas_ciudad_estado, conn)
    df_q4 = pd.read_sql(sql_q4_vendedor_mejor_ticket_promedio, conn)
    df_q5 = pd.read_sql(sql_q5_productos_descuento_y_perdida_margen, conn)

print('Q1) Ventas por mes, categoria y cliente')
display(df_q1.head(20))

print('Q2) Clientes con mayor margen')
display(df_q2.head(20))

print('Q3) Evolucion de ventas por ciudad y estado')
display(df_q3.head(20))

print('Q4) Vendedor con mejor ticket promedio')
display(df_q4.head(20))

print('Q5) Productos con perdida de margen por descuentos frecuentes')
display(df_q5.head(20))

Q1) Ventas por mes, categoria y cliente


,anio_numero,mes_numero,mes_nombre,categoria_cliente,nombre_cliente,ventas_netas,unidades_vendidas,margen_bruto
0,2013,1,Enero,Computer Store,Hoc Tran,21117.45,98.0,8434.50
1,2013,1,Enero,Computer Store,Emil Roman,13334.71,409.0,5862.50
2,2013,1,Enero,Computer Store,Madhu Dwivedi,11244.88,815.0,4223.35
3,2013,1,Enero,Computer Store,Olya Izmaylov,10502.03,371.0,4551.10
4,2013,1,Enero,Computer Store,Kristiina Ivanov,9177.17,446.0,3852.70
5,2013,1,Enero,Computer Store,Jitka Necesana,7490.81,442.0,2684.25
6,2013,1,Enero,Computer Store,Anindya Ghatak,7002.70,795.0,3150.80
7,2013,1,Enero,Computer Store,Gopalgobinda Sikdar,6852.05,342.0,2300.80
8,2013,1,Enero,Computer Store,Mahavir Sonkar,6828.13,696.0,3637.50
9,2013,1,Enero,Computer Store,Kertu Sokk,6629.75,274.0,3227.50


Q2) Clientes con mayor margen


,id_cliente_nk,nombre_cliente,margen_bruto_total,ventas_netas_totales,margen_pct
0,149,"Tailspin Toys (Inguadona, MN)",189277.45,438689.81,0.431460
1,964,Ingrida Zeltina,189239.35,413817.88,0.457301
2,580,"Wingtip Toys (Sarversville, PA)",188919.75,420001.08,0.449807
3,132,"Tailspin Toys (Minidoka, ID)",185432.55,427445.57,0.433815
4,472,"Wingtip Toys (San Jacinto, CA)",185158.65,408524.95,0.453237
5,954,Nasrin Omidzadeh,181555.60,416005.03,0.436426
6,510,"Wingtip Toys (Grabill, IN)",180712.10,403635.05,0.447711
7,856,Satish Mittal,179484.75,397571.38,0.451452
8,593,"Wingtip Toys (Cuyamungue, NM)",178329.95,405824.67,0.439426
9,949,Seo-yun Paik,178316.20,393305.81,0.453377


Q3) Evolucion de ventas por ciudad y estado


,anio_numero,mes_numero,estado_provincia,ciudad,ventas_netas,margen_bruto,unidades_vendidas
0,2013,1,Alabama,Highland Home,39397.52,18500.60,1286.0
1,2013,1,Alabama,Marion Junction,14656.29,6334.00,180.0
2,2013,1,Alabama,Tuscaloosa,14374.08,6374.50,1156.0
3,2013,1,Alabama,Eulaton,11902.50,5220.25,458.0
4,2013,1,Alabama,Robertsdale,11651.34,6025.60,643.0
5,2013,1,Alabama,Guin,9232.20,4146.00,310.0
6,2013,1,Alabama,Bazemore,8186.97,3658.90,468.0
7,2013,1,Alabama,Broomtown,6663.79,2316.50,351.0
8,2013,1,Alabama,Jemison,6222.65,3035.50,150.0
9,2013,1,Alabama,Flomaton,5790.25,2115.00,185.0


Q4) Vendedor con mejor ticket promedio


,id_vendedor_nk,nombre_completo,ticket_promedio,cantidad_facturas,ventas_netas_totales
0,16,Archer Lamble,2871.631881,7184,20629803.44
1,3,Hudson Onslow,2848.710064,6988,19906785.93
2,2,Kayla Woodcock,2820.847955,7160,20197271.36
3,14,Lily Code,2816.126837,6978,19650933.07
4,15,Taj Shand,2810.648027,7074,19882524.15
5,6,Sophia Hinton,2808.712495,7073,19866023.48
6,13,Hudson Hollinworth,2782.576314,7079,19697857.73
7,20,Jack Potter,2779.825996,7061,19628351.36
8,8,Anthony Grosse,2774.033704,6954,19290630.38
9,7,Amy Trefl,2772.418242,6959,19293258.55


Q5) Productos con perdida de margen por descuentos frecuentes


,id_producto_nk,nombre_producto,marca,lineas_totales,lineas_con_descuento,pct_lineas_con_descuento,perdida_ingreso_estimada,margen_bruto_total
0,215,Air cushion machine (Blue),NaN,1061,1061,100.0,5498118.49,4439391.0
1,173,32 mm Anti static bubble wrap (Blue) 50m,NaN,1085,1085,100.0,3160384.00,3526400.0
2,167,10 mm Anti static bubble wrap (Blue) 50m,NaN,1119,1119,100.0,3133209.30,3452220.0
3,161,20 mm Double sided bubble wrap 50m,NaN,1059,1059,100.0,3076088.40,5293680.0
4,164,32 mm Double sided bubble wrap 50m,NaN,1004,1004,100.0,3064168.80,2929310.0
5,158,10 mm Double sided bubble wrap 50m,NaN,1035,1035,100.0,2942068.00,2773400.0
6,170,20 mm Anti static bubble wrap (Blue) 50m,NaN,1033,1033,100.0,2868841.80,2670540.0
7,172,32 mm Anti static bubble wrap (Blue) 20m,NaN,1088,1088,100.0,1435579.20,1510500.0
8,219,Void fill 400 L bag (White) 400L,NaN,1039,1039,100.0,1421392.50,1378320.0
9,169,20 mm Anti static bubble wrap (Blue) 20m,NaN,994,994,100.0,1222280.80,1316640.0
